# Suspended IDs

This notebook grabs additional data to flesh out suspect user IDs. PRAW is not available for suspended accounts, meaning we need to source data elsewhere. This process is different than the active account info collection because you can use Reddit.

In [ ]:
# ---- Libraries -----------------------------------------------------------------
import pandas as pd
import numpy as np
import os

# Show every column/row when inspecting a frame - project convention.
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

import re
from string import punctuation
import functools
import operator
from dotenv import load_dotenv
import praw                                  # unused here: PRAW cannot read suspended users
import prawcore
from prawcore.exceptions import NotFound
import pprint
import ast
import pprint
import os
import itertools
import pickle
import requests                              # HTTP calls to the Arctic Shift API
from time import time
from requests_ratelimiter import LimiterSession   # caps request rate (see get_historic_data)
import pprint
import json                                  # decodes the Arctic Shift JSON payloads
import base64
from itertools import islice
import pyarrow as pa                         # parquet engine

In [ ]:
# Accounts produced by suspicious_id_collection_v1.ipynb: one row per suspicious user,
# carrying the account status and first-mention date.
users = pd.read_parquet('User_Profiles.parquet', engine='pyarrow')

In [ ]:
# Small slice used to exercise the collection function before running the full list.
u = users['username'][0:5]

In [ ]:
def get_historic_data(name_list):
    """
    Pull every submission Arctic Shift holds for each account in `name_list`.

    Arctic Shift is a Pushshift successor and, unlike PRAW, still serves the post history
    of suspended accounts - which is the whole reason this notebook exists.
    Returns one flat DataFrame: one row per post, with the per-post `_meta` block merged
    into the row and an `outer_key` column naming the account it came from.
    """
    # 5 requests/second, enforced by the session itself so no manual sleep is needed.
    session = LimiterSession(per_second=5)
    user_submissions = {}

    for name in name_list:
        print(name)
        all_posts = []
        after = None          # pagination cursor: created_utc of the last post seen

        while True:
            # limit=100 is the API maximum per page.
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/posts/search?author={name}&limit=100'
            if after:
                arctic_url += f'&after={after}'

            retries = 0
            success = False

            # Retry loop: up to 10 attempts for this one page.
            while retries < 10:
                print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)

                    if response.status_code == 200:
                        # Reach into the response object for the raw bytes, then decode.
                        # (response.json() would do the same thing more directly.)
                        api_return = response.__dict__
                        data_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))

                        posts = data_content['data']
                        all_posts.extend(posts)

                        # A short page means the account's history is exhausted.
                        if len(posts) < 100:
                            success = True
                            break

                        # Full page: advance the cursor so the next request continues after
                        # the last post returned.
                        after = posts[-1]['created_utc']
                        success = True
                        break

                    else:
                        # Non-200: count the attempt and try the same page again.
                        retries += 1
                        print(f'API response: {response.status_code}')

                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    raise e

            # BUG (left as-is, flagged not fixed - fixing it changes what data gets collected):
            # this stops paginating precisely when a FULL page came back, i.e. when there is
            # more history to fetch, so only the first 100 posts per account are ever kept.
            # The condition is inverted; it should continue while len(posts) == 100 and stop
            # on a short page. Note also that `posts` is unbound here if all 10 retries fail.
            if not success or len(posts) == 100:
                break

        user_submissions[name] = all_posts

    # Build DataFrame from all users
    # Flatten {account: [post, ...]} into one row per post. The `_meta` sub-dict is merged
    # up into the row so its fields become ordinary columns; the "|" dict-merge requires
    # Python 3.9+.
    rows = []
    for outer_key, records in user_submissions.items():
        for record in records:
            meta = record.get('_meta', {})
            row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
            rows.append(row)

    df = pd.DataFrame(rows)

    return df

In [ ]:
# Trial run against the 5-account slice.
test=get_historic_data(u)

In [ ]:
# Confirm a DataFrame came back rather than an error or empty result.
type(test)

In [ ]:
# Distinct submissions collected - compare against .shape below to spot duplicate rows.
test.id.nunique()

In [ ]:
# (rows, columns) of the collected history.
test.shape

In [ ]:
# Drop columns holding dicts/lists: unhashable values break duplicated() below, and they
# cannot be written to parquet either. .map over the frame tests every cell, and .any()
# marks a column for removal if it holds any nested value.
t = test.loc[:, ~test.map(lambda x: isinstance(x, (dict,list))).any()]

In [ ]:
# Count fully duplicated rows across the flat columns - a check on the pagination logic,
# since a cursor that fails to advance would re-collect the same page.
t.duplicated().sum()

In [ ]:
# Scratch arithmetic left from manually reconciling collected post counts.
99 + 132